In [ ]:
course:
    name: "CI/CD for Machine Learning"
    prerequisites:
        - "MLOps Concepts"
        - "Supervised Learning with scikit-learn"
        - "Introduction to Git"

In [ ]:
file:///home/repl/workspace/course_information.yaml

In [ ]:
courses:
  - name: Machine Learning 101
    # Complete prerequisites in block format
    prerequisites:
      - Linear Algebra
      - Python Programming
      - Statistics
    # Write key for students
    students:
      - name: John Doe
        # Write midterm scores in flow format
        midterm_scores: [85, 92, 78]
        final_score: 88
      - name: Jane Smith
        # Complete midterm scores in block format
        midterm_scores:
          - 78
          - 84
          - 90
        final_score: 92

In [ ]:
file:///home/repl/workspace/python_workflow.yaml

In [ ]:
config:
  slack:
    channels:
      - workflow-orchestration
      - builds-911

workflow:
  # Use the correct block style indicator 
  # while writing commands to run
  run: |
    echo "Running script.py"
    python3 script.py
  notify:
    - slack:
        # Reference the Slack channels using 
        # placeholder defined in config block
        channels: ${{ config.slack.channels }}
        # Use the correct block style indicator 
        # to send the message without newlines
        message: >-
          It appears that your run has failed.
          To gain more insights into the failure,
          it is recommended to examine the CI logs.
          In case further assistance is required,
          feel free to contact the Engineer on call.
      if: run.state == "failed"

In [ ]:
file:///home/repl/workspace/.github/workflows/push_workflow.yaml

In [ ]:
name: CI

on:
  push:
    # Specify the branch that will trigger the workflow
    branches: [ "myfeature" ]

jobs:
  # Write name of the job as a key
  hello-from-runner:
    # Specify the runner machine
    runs-on: ubuntu-22.04
    steps:
      # Write the step name
      - name: Print Name
        run: |
          echo "Hello from $(whoami)"

In [ ]:
name: CI

on:
  # Specify the event that will trigger the workflow
  pull_request:
    branches: [ "main" ]

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      # Use checkout action to check out the repository on the runner
      - name: Checkout
        uses: actions/checkout@v3

      # Setup Python to 3.9 using setup-python action
      - name: Setup Python
        uses: actions/setup-python@v4
        with:
          python-version: 3.9

      # Run the hello_world.py python script
      - name: Run Python script
        run: |
          python3 hello_world.py

In [ ]:
file:///home/repl/workspace/print_environment_variables.yaml

In [ ]:
name: Testing Environment variables

on:
  pull_request:
    # Write the target branch
    branches: main

env:
  # Set the value of global environment variable
  GLOBAL_VARIABLE: global_value

jobs:
  
  # Job block
  print_env_and_secrets:
    runs-on: ubuntu-latest
    
    env:
      # Set the value of local environment variable
      JOB_VARIABLE: job_value

    steps:
      - name: Print Variables
        # Write the environment variable whose value is set
        run: |
          echo "Global Variable: ${{ env.GLOBAL_VARIABLE }}"
          echo "Set job Variable: ${{ env.JOB_VARIABLE }}"

In [ ]:
file:///home/repl/workspace/secrets.yaml

In [ ]:
name: Testing Secrets

on:
  # Write the event triggering the workflow
  pull_request:
    branches: master

jobs:
  print_secrets:
    runs-on: ubuntu-latest
    
    permissions: 
      # Grant the write permissions
      pull-requests: write

    steps:
      - name: Comment on Pull Request
        uses: thollander/actions-comment-pull-request@v2
        with:
          # Access the GITHUB_TOKEN token from secrets
          GITHUB_TOKEN: ${{ secrets.GITHUB_TOKEN }}
          message: |
            Hello world ! :wave:

In [ ]:
import json

import pandas as pd
from sklearn.model_selection import train_test_split

from metrics_and_plots import plot_confusion_matrix, save_metrics
from model import evaluate_model, train_model
from utils_and_constants import PROCESSED_DATASET, TARGET_COLUMN


def load_data(file_path):
    data = pd.read_csv(file_path)
    X = data.drop(TARGET_COLUMN, axis=1)
    y = data[TARGET_COLUMN]
    return X, y


def main():
    X, y = load_data(PROCESSED_DATASET)
    
    # Split data into training and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1993)

    # Train the model using the training set
    model = train_model(X_train, y_train)
    
    # Calculate test set metrics
    metrics = evaluate_model(model, X_test, y_test)

    print("====================Test Set Metrics==================")
    print(json.dumps(metrics, indent=2))
    print("======================================================")

    # Save metrics into json file
    save_metrics(metrics)
    plot_confusion_matrix(model, X_test, y_test)


if __name__ == "__main__":
    main()

In [ ]:
name: model-training

on:
  pull_request:
    branches: main

permissions: write-all

jobs:
  train_and_report_eval_performance:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout 
        uses: actions/checkout@v3
      
      - name: Setup Python
        uses: actions/setup-python@v4
        with:
          python-version: 3.9

      # Setup CML GitHub Action
      - name: Setup CML
        uses: iterative/setup-cml@v1
          
      - name: Train model
        run: |
          python3 preprocess_dataset.py
          python3 train.py

      - name: Write CML report
        env:
          REPO_TOKEN: ${{ secrets.GITHUB_TOKEN }}
        run: |
          # Add metrics data to markdown
          cat metrics.json >> model_eval_report.md
          
          # Add confusion matrix plot to markdown
          echo "![confusion matrix plot](./confusion_matrix.png)" >> model_eval_report.md

          # Create comment from markdown report
          cml comment create model_eval_report.md

In [ ]:
file:///home/repl/workspace/dvc.yaml

In [ ]:
stages:
  preprocess:
    # Run the data preprocessing script
    cmd: python3 preprocess_dataset.py
    deps:
    - preprocess_dataset.py
    - raw_dataset/weather.csv
    - utils_and_constants.py
    outs:
    - processed_dataset/weather.csv
  train:
    # Run the model training script
    cmd: python3 train.py
    deps:
    - metrics_and_plots.py
    - model.py
    # Specify the preprocessed dataset as a dependency
    - processed_dataset/weather.csv
    - train.py
    - utils_and_constants.py
    outs:
    - metrics.json
    - confusion_matrix.png

In [ ]:
stages:
  preprocess:
    cmd: python3 preprocess_dataset.py
    deps:
    - preprocess_dataset.py
    - raw_dataset/weather.csv
    - utils_and_constants.py
    outs:
    - processed_dataset/weather.csv
  train:
    cmd: python3 train.py
    deps:
    - metrics_and_plots.py
    - model.py
    - processed_dataset/weather.csv
    - train.py
    - utils_and_constants.py
    metrics:
      # Specify the metrics file as target
      - metrics.json:
          cache: false
    plots:
      # Set the target to the file containing predictions data
      - predictions.csv:
          # Write the plot template
          template: confusion_normalized
          x: predicted_label
          y: true_label
          x_label: 'Predicted label'
          y_label: 'True label'
          title: Confusion matrix
          # Set the cache parameter to store
          # plot data in git repository
          cache: false

In [ ]:
import shutil
from pathlib import Path

DATASET_TYPES = ["test", "train"]
DROP_COLNAMES = ["Date"]
TARGET_COLUMN = "RainTomorrow"
RAW_DATASET = "raw_dataset/weather.csv"
PROCESSED_DATASET = "processed_dataset/weather.csv"
RFC_FOREST_DEPTH = 4


def delete_and_recreate_dir(path):
    try:
        shutil.rmtree(path)
    except:
        pass
    finally:
        Path(path).mkdir(parents=True, exist_ok=True)

In [ ]:
name: dvc-pipeline

on:
  pull_request:
    branches: main

permissions: write-all

jobs:
  train_and_report_eval_performance:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout 
        uses: actions/checkout@v3
      
      - name: Setup Python
        uses: actions/setup-python@v4
        with:
          python-version: 3.9

      - name: Setup CML
        uses: iterative/setup-cml@v1
          
      # Setup DVC GitHub Action
      - name: Setup DVC
        uses: iterative/setup-dvc@v1
          
      # Run DVC pipeline
      - name: Run DVC pipeline
        run: dvc repro

      - name: Write CML report
        env:
          REPO_TOKEN: ${{ secrets.GITHUB_TOKEN }}
        run: |
          # Compare metrics with main branch
          git fetch --prune
          dvc metrics diff --md main >> metrics_compare.md
          
          # Create comment from markdown report
          cml comment create metrics_compare.md

In [ ]:
file:///home/repl/workspace/hp_tuning.py

In [ ]:
import json

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from utils_and_constants import PROCESSED_DATASET, get_hp_tuning_results, load_data


def main():
    X, y = load_data(PROCESSED_DATASET)
    X_train, _, y_train, _ = train_test_split(X, y, random_state=1993)

    model = RandomForestClassifier()
    # Read the config file to define the hyperparameter search space
    param_grid = json.load(open("hp_config.json", "r"))

    # Perform Grid Search Cross Validation on training data
    grid_search = GridSearchCV(model, param_grid, cv=5, n_jobs=1, verbose=2)
    grid_search.fit(X_train, y_train)

    best_params = grid_search.best_params_

    print("====================Best Hyperparameters==================")
    print(json.dumps(best_params, indent=2))
    print("==========================================================")

    with open("rfc_best_params.json", "w") as outfile:
        json.dump(best_params, outfile)

    markdown_table = get_hp_tuning_results(grid_search)
    with open("hp_tuning_results.md", "w") as markdown_file:
        markdown_file.write(markdown_table)


if __name__ == "__main__":
    main()

In [ ]:
name: hp-tuning

on:
  pull_request:
    branches: main

permissions: write-all

jobs:
  hp_tune:
    # Only run job if the current repository
    # starts with the right prefix
    if: startsWith(github.head_ref, 'hp_tune/')
    runs-on: ubuntu-latest
    steps:
      - name: Checkout 
        uses: actions/checkout@v3
      
      - name: Setup Python
        uses: actions/setup-python@v4
        with:
          python-version: 3.9

      - name: Setup DVC
        uses: iterative/setup-dvc@v1

      - name: Setup CML
        uses: iterative/setup-cml@v1

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run DVC pipeline
        run: |
          dvc repro -f hp_tune
      
      - name: Create training branch
        env:
          REPO_TOKEN: ${{ secrets.GITHUB_TOKEN }}
        run: |
          # Finish the create pull request command
          cml pr create \
          --user-email hp-bot@cicd.ai \
          --user-name HPBot \
          --message "HP tuning" \
          # Write the new head branch name
          --branch train/${{ github.sha }} \
          # Write the target branch name
          --target-branch main \
          # Commit the hyperparameter job output file
          rfc_best_params.json